# GTR + RADAR v5 on Colab (Python 3.12, single A100, fp32 + TF32)

1. Runtime → Change runtime type → **A100 GPU**.
2. Push the repo to GitHub (or upload it to Drive).
3. Put the original dataset files in a Drive folder (`ETTh1.csv`, `ETTh2.csv`, `ETTm1.csv`, `ETTm2.csv`, `weather.csv`, `electricity.csv`, `traffic.csv`). Nothing is generated synthetically; a missing file stops the pipeline.

Order of the v4 plan: **step 1** RevIN 2×2 → **step 2** offline experts → **step 3** detector features → **step 4** full grid.
Checkpoints are reused when they exist (`SKIP_EXISTING=1`): `base.pth` (pure GTR) from earlier runs is kept; every RADAR / GTR+Aug checkpoint uses new `*_v4` names. Results go to `results_radar_v5/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
import torch, numpy, pandas, sklearn
print('torch', torch.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/knull-cc/GTR_RADAR.git'      # or copy the repo from Drive instead
DATA_SRC = '/content/drive/MyDrive/datasets'                 # folder holding the dataset files
SAVE_DIR = '/content/drive/MyDrive/GTR_RADAR_runs'           # results / figures / logs / checkpoints

In [ ]:
%cd /content
!test -d GTR_RADAR || git clone $REPO_URL GTR_RADAR
%cd /content/GTR_RADAR
!git log --oneline -1
!mkdir -p dataset && cp -rn "$DATA_SRC"/* dataset/ 2>/dev/null; ls -R dataset | head -40
import os
os.environ['CKPT_DIR'] = SAVE_DIR + '/checkpoints_radar/'   # survives disconnects; base.pth is reused from here

## 0 · Unit tests (B=4, L=96, C=7, H=96; random GTR in eval mode)

In [ ]:
!pip -q install pytest
!python -m pytest -q tests/test_radar.py

## 1 · Step 1 — RevIN 2×2 on B0 (validation split, H=96)
Clean / Raw / Stats-only / Direct-only forecast MSE per type × position. Decides the gate / dependency branch:
Internal Raw ≈ Clean → keep the gate; damage mostly *Stats-only* → add the repair-size term; mostly *Direct-only* → per-sample dependency.

In [ ]:
!bash scripts/RADAR/run_step.sh revin ETTh1
from IPython.display import Markdown, display
import glob
for f in sorted(glob.glob('results_radar_v5/diag/*__revin.md')):
    display(Markdown(open(f).read()))

## 2 · Step 2 — offline experts (oracle detection, no training)
Repair MAE of no-repair / local median / E2 v3 / E2 v4 / E3 / E3-else-E2 / best-of-E2-E3 on the corrupted points, and the B0 forecast MSE after replacement.
ETTh1/ECL/Traffic have E3 (T = 24); ETTm (T = 96) and Weather (T = 144) do not (T ≥ L = 96).

In [ ]:
!bash scripts/RADAR/run_step.sh experts ETTh1
for f in sorted(glob.glob('results_radar_v5/diag/*__experts.md')):
    display(Markdown(open(f).read()))

## 3 · Step 3 — detector features v4 vs v3 (S2 only, AUROC per type × position)

In [ ]:
!bash scripts/RADAR/run_step.sh det ETTh1
!DET_FEATS=v3 bash scripts/RADAR/run_step.sh det ETTh1
for f in sorted(glob.glob('results_radar_v5/diag/*__det_det*.md')):
    display(Markdown(open(f).read()))

## 4 · Step 4 — main experiment
Quick check on ETTh1 first (H=96), then the 7 datasets × H ∈ {96, 192, 336, 720}, seed 2024.
`RADAR_ARGS="--test_grid main"` halves the evaluation time (100 % channels only); `ARGMAX=1` also evaluates hard routing.

In [ ]:
!REPRO=0 bash scripts/RADAR/run_quick.sh ETTh1   # REPRO=1 re-runs the original run.py check
display(Markdown(open('results_radar_v5/summary.md').read()))

Add the v4 plug-in (already trained, `plugin_v4.pth`) to the same results as **Ours-plugin-v4** for a direct comparison:

In [ ]:
!ARCH=v4 METHODS=Ours-plugin MODE=eval QUICK=1 REPRO=0 bash scripts/RADAR/etth1.sh
display(Markdown(open('results_radar_v5/summary.md').read()))

In [ ]:
!bash scripts/RADAR/run_all.sh etth1 etth2 ettm1 ettm2
!mkdir -p "$SAVE_DIR" && cp -r results_radar_v5 figures_radar_v5 logs_radar_v5 "$SAVE_DIR"/

In [ ]:
!bash scripts/RADAR/run_all.sh weather electricity traffic
!mkdir -p "$SAVE_DIR" && cp -r results_radar_v5 figures_radar_v5 logs_radar_v5 "$SAVE_DIR"/
display(Markdown(open('results_radar_v5/summary.md').read()))
display(Markdown(open('results_radar_v5/summary_router.md').read()))